# ipm_chol vs ipm_chol_sparse — timing comparison

Load results from `sparse_vs_dense_results.pkl` and plot wall-time as a function of `d` across a grid of average graph degrees.

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

RESULTS_FILE = Path('sparse_vs_dense_results.pkl')

with open(RESULTS_FILE, 'rb') as f:
    raw = pickle.load(f)

df = pd.DataFrame(raw)
print(f"{len(df)} rows, d values: {sorted(df['d'].unique())}")
print(f"avg_degree values: {sorted(df['avg_degree'].unique())}")
df.head()

## 1. Time vs d — one line per avg_degree, side-by-side solvers

In [ ]:
avg_degrees = sorted(df['avg_degree'].unique())
d_vals      = sorted(df['d'].unique())
cmap        = plt.cm.viridis
colors      = [cmap(i / (len(avg_degrees) - 1)) for i in range(len(avg_degrees))]

# Aggregate over seeds
agg = df.groupby(['d', 'avg_degree'])[['time_dense', 'time_sparse']].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, col, title in zip(axes,
                           ['time_dense',  'time_sparse'],
                           ['ipm_chol (dense)', 'ipm_chol_sparse']):

    # -- Reference lines for O(d), O(d²), O(d³) anchored at d=128 --
    d_ref = np.array(sorted(df['d'].unique()), dtype=float)
    anchor_d = 128.0
    anchor_t = 0.5   # approximate visual anchor in seconds
    for exp, ls, lbl in [(1, (0,(5,3)), 'O(d)'), (2, (0,(3,2)), 'O(d²)'), (3, (0,(1,2)), 'O(d³)')]:
        scale = anchor_t / anchor_d**exp
        ax.plot(d_ref, scale * d_ref**exp, color='gray', lw=1.0, ls=ls,
                alpha=0.45, label=lbl, zorder=0)

    # -- Data + fit lines --
    for deg, color in zip(avg_degrees, colors):
        sub = agg[agg['avg_degree'] == deg].dropna(subset=[col])
        xs = sub['d'].values.astype(float)
        ys = sub[col].values
        if len(xs) < 2:
            ax.plot(xs, ys, marker='o', color=color, label=f'deg={deg:.1f}')
            continue

        slope, intercept = np.polyfit(np.log(xs), np.log(ys), 1)
        ax.plot(xs, ys, marker='o', color=color, label=f'deg={deg:.1f}  (α={slope:.2f})')

        x_fit = np.geomspace(xs.min(), xs.max(), 120)
        ax.plot(x_fit, np.exp(intercept) * x_fit**slope,
                color=color, lw=1.2, ls='--', alpha=0.55)

    ax.set_xscale('log', base=2)
    ax.set_yscale('log')
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x)}'))
    ax.set_xlabel('d')
    ax.set_ylabel('Wall time (s)')
    ax.set_title(title)
    ax.legend(fontsize=7.5, ncol=1)
    ax.grid(True, which='both', alpha=0.3)

fig.suptitle('Solver wall time vs problem size  (dashed = power-law fit, grey = reference slopes)', fontsize=12)
fig.tight_layout()
plt.show()

## 2. Speedup ratio (dense / sparse) vs d

In [ ]:
agg2 = agg.copy()
agg2['speedup'] = agg2['time_dense'] / agg2['time_sparse']

fig, ax = plt.subplots(figsize=(8, 5))

for deg, color in zip(avg_degrees, colors):
    sub = agg2[(agg2['avg_degree'] == deg) & agg2['speedup'].notna()]
    ax.plot(sub['d'], sub['speedup'], marker='o', color=color, label=f'deg={deg:.1f}')

ax.axhline(1.0, color='k', lw=0.8, ls='--', label='break-even')
ax.set_xscale('log', base=2)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x)}'))
ax.set_xlabel('d')
ax.set_ylabel('Speedup  (time_dense / time_sparse)')
ax.set_title('Speedup of ipm_chol_sparse over ipm_chol_dense')
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()
plt.show()

## 3. Speedup heatmap — d × avg_degree

In [ ]:
pivot = agg2.pivot(index='avg_degree', columns='d', values='speedup')

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn',
               vmin=0.5, vmax=pivot.values[np.isfinite(pivot.values)].max())

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([str(c) for c in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f'{v:.1f}' for v in pivot.index])
ax.set_xlabel('d')
ax.set_ylabel('avg_degree')
ax.set_title('Speedup  (dense / sparse)  —  green > 1 means sparse wins')

# Annotate cells
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        if np.isfinite(val):
            ax.text(j, i, f'{val:.1f}x', ha='center', va='center',
                    fontsize=8, color='black')
        else:
            ax.text(j, i, '—', ha='center', va='center', fontsize=9, color='grey')

fig.colorbar(im, ax=ax, label='Speedup')
fig.tight_layout()
plt.show()

## 4. Time vs avg_degree — one line per d (sparse solver only, includes d=2048)

In [ ]:
d_cmap   = plt.cm.plasma
d_colors = [d_cmap(i / (len(d_vals) - 1)) for i in range(len(d_vals))]

fig, ax = plt.subplots(figsize=(8, 5))

for d, color in zip(d_vals, d_colors):
    sub = agg[agg['d'] == d].dropna(subset=['time_sparse'])
    ax.plot(sub['avg_degree'], sub['time_sparse'], marker='o',
            color=color, label=f'd={d}')

ax.set_xlabel('avg_degree')
ax.set_ylabel('Wall time (s)')
ax.set_yscale('log')
ax.set_title('ipm_chol_sparse — time vs avg_degree')
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()
plt.show()